In [1]:
import torch

from rlaopt.linalg import IdentityConfig, LinSys, NystromConfig
from rlaopt.solvers import PCG, PCGConfig, PCGStoppingCriteria

In [2]:
torch.set_default_dtype(torch.float64)

In [3]:
n = 10000
eigvals = torch.arange(1, n + 1) ** -2.0
reg = 1e-3

U = torch.randn(n, n)
U = torch.linalg.qr(U).Q

A = U @ torch.diag(eigvals) @ U.T
B = torch.randn(n, 10)

In [4]:
lin_sys = LinSys(A, B, reg)
lin_sys.cuda()

LinSys()

In [5]:
preconditioner_config_identity = IdentityConfig()
preconditioner_config_nystrom = NystromConfig(rank=100, base_damping=reg)

In [6]:
solver_config = PCGConfig(
    preconditioner_config=preconditioner_config_identity,
)

In [7]:
solver = PCG(lin_sys, solver_config)
params = lin_sys.w.clone()
state = solver.init_state(params)

In [8]:
max_iters = 100

for i in range(max_iters):
    params, state = solver.step(params, state)
    print(f"Iteration {i + 1}, residual norm: {state.res_norm}")

Iteration 1, residual norm: tensor([ 340.3818,  217.6764,  613.2061,  311.4263,  206.7340,  299.5945,
         611.6343,  280.4595, 1242.4448,  272.2981], device='cuda:0',
       grad_fn=<LinalgVectorNormBackward0>)
Iteration 2, residual norm: tensor([12.7887, 10.8983,  7.7300,  8.4673, 13.3490, 17.6615, 16.4376,  9.2218,
        21.5994, 17.4729], device='cuda:0',
       grad_fn=<LinalgVectorNormBackward0>)
Iteration 3, residual norm: tensor([3.1016, 2.5565, 2.8370, 1.7912, 3.0505, 5.2689, 3.8231, 2.6881, 3.8415,
        3.4889], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 4, residual norm: tensor([0.7004, 0.5529, 0.6438, 0.4833, 0.7276, 1.1259, 0.9990, 0.6582, 1.1918,
        0.9083], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 5, residual norm: tensor([0.1605, 0.1088, 0.1401, 0.0942, 0.1382, 0.2129, 0.1859, 0.1367, 0.3084,
        0.2359], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 6, residual norm: tensor([0.1643, 0.0859,

In [9]:
params, final_res_norm = solver.solve(
    stopping_criteria=PCGStoppingCriteria(max_iters=100, tol=1e-10)
)
print(f"Final residual norm: {final_res_norm}")

Final residual norm: tensor([5.7047e-09, 6.6984e-09, 7.7871e-09, 5.5257e-09, 7.3156e-09, 9.8111e-09,
        8.7220e-09, 5.0323e-09, 5.5394e-09, 6.4831e-09], device='cuda:0',
       grad_fn=<LinalgVectorNormBackward0>)


In [10]:
lin_sys.compute_residual_norm(params, relative=True)

tensor([5.7607e-11, 6.6727e-11, 7.8254e-11, 5.4938e-11, 7.3646e-11, 9.9436e-11,
        8.7409e-11, 5.0049e-11, 5.4941e-11, 6.4579e-11], device='cuda:0',
       grad_fn=<DivBackward0>)